### Vector Stores 
Let's learn vector store directly by implemention a project **Movie Recommendation System**

#### 1. Loading the json file 

In [ ]:
import json
from langchain_core.documents import Document

with open("../1_Datasets/movies_data_5000.json", "r", encoding="utf-8") as f:
    data = json.load(f)

final_docs = [Document(**item) for item in data]
final_docs[:4]

We don't need to perform splitting(chunking) here becasue **CSVLoader** load each row as one document object. But we need to preprocess each document.

#### 2. Loading the Embedding Model

In [3]:
# We will load the embedding model from huggingface
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "Sentence-transformers/all-MiniLM-L6-v2"
)

d:\Generative AI using LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1074.36it/s]


#### 3. Creating & Loading the Chroma Vector Store 

In [5]:
!uv add chromadb -q

In [6]:
from langchain_community.vectorstores import Chroma

# Create Chroma vector store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db",
    collection_name="movies"
)

#### 4. Adding the documents to Chroma Vector Store

In [7]:
# Well this part will take time 
from tqdm import tqdm # To add progress bar 

for i in tqdm(range(0, len(final_docs)),desc="Indexing Movies"):
    vector_store.add_documents([final_docs[i]])

Indexing Movies: 100%|██████████| 5000/5000 [20:03<00:00,  4.15it/s]  


#### 5. Searching the similar documents 


In [8]:
query = """
An 11-year-old boy discovers he is a wizard and begins studying at a magical boarding school. 
As he learns magic and forms lasting friendships, he uncovers the truth about his family's past
and confronts a powerful dark wizard.
"""
similar_docs = vector_store.similarity_search(
    query=query,
    k=10
)

In [9]:
# Extracting the titles of the movies
for doc in similar_docs: 
    print(doc.metadata["title"])

 Harry Potter and the Philosopher's Stone
 Harry Potter and the Order of the Phoenix
 The Sword in the Stone
 Harry Potter and the Goblet of Fire
 Bogus
 About a Boy
 Harry Potter and the Prisoner of Azkaban
 The Craft
 Journey to the Beginning of Time
 My Tutor


#### 6. Pro Tip: Defining the function 

In [10]:
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

def recommend_movies(query, num_movies=10):
    similar_docs = vector_store.similarity_search(
    query=query,
    k=num_movies
    )
    
    movies = [f"{TMDB_IMAGE_BASE_URL}{doc.metadata['poster_path'].strip()}" for doc in similar_docs]
    return movies

#### 7. Let's make a frontend with gradio

In [12]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo: 

    gr.Markdown("# Movie Recommendation System")

    movie_input = gr.Textbox(
        label="Movie Description", 
        placeholder="Enter a movie title or description..."
        )
    
    search_button = gr.Button("Search similar Movies", variant="primary")

    gallery = gr.Gallery(
        label = "Recommended Movies", 
        columns=5, 
        height=500
    )

    search_button.click(
        fn = recommend_movies, 
        inputs = movie_input, 
        outputs = gallery
    ) 

demo.launch()

C:\Users\CSE\AppData\Local\Temp\ipykernel_29408\1133889584.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


d:\Generative AI using LangChain\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
